# DIVE-lite Robot Failure Watcher (open-source)
**Problem (from Chelsea Finn's keynote):** VLAs like π0.5 fail silently in unseen homes (incomplete placement, driving over clothes, missed thin-object grasps, semantically wrong actions).
**Method (inspired by the DIVE CVRR-challenge talk + two-stage verification):** intent estimation → question breakdown → object-centric summary (Grounding DINO) → iterative QA with continuation judgment → verdict → verification.
**Comparison:** one-shot Video-LLM baseline vs. DIVE-lite, per failure category, plus latency.

Runtime: Colab → Runtime → Change runtime type → **T4 GPU**.

In [3]:
!pip -q install "transformers>=4.51" accelerate "qwen-vl-utils[decord]" opencv-python-headless pandas matplotlib

## 1. Config
Clips live in **My Drive/vids** as `.MOV` files. The labels come from the filenames:
anything containing `_success` is a success, and everything else is a failure. The part after the task name (e.g. `wrongpocket`, `lanyardvisible`) is stored as the failure type.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os

CLIP_DIR = "/content/drive/MyDrive/vids"               # your .MOV clips
MP4_DIR  = "/content/clips_mp4"                        # converted copies (local Colab disk)
OUT_DIR  = "/content/drive/MyDrive/vla_watcher_results"
os.makedirs(MP4_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
CSV_PATH = f"{OUT_DIR}/clips.csv"
OUT_CSV  = f"{OUT_DIR}/results.csv"

VLM_ID  = "Qwen/Qwen2.5-VL-3B-Instruct"   # fits a free T4 in fp16
DINO_ID = "IDEA-Research/grounding-dino-tiny"

FPS = 1.0                 # frame sampling rate sent to the Video-LLM
MAX_PIXELS = 360 * 420    # per-frame resolution cap; lower it if you run out of memory
DINO_FRAMES = 6           # frames sampled for the object-centric summary
MAX_ROUNDS = 2            # continuation-judgment rounds in DIVE-lite

TASKS = {
    "bottle": ("put the water bottle in the backpack's outermost open side pocket",
               "water bottle. backpack. pocket."),
    "keys":   ("put the keys inside the backpack and zip it completely",
               "keys. lanyard. backpack. zipper."),
}

Mounted at /content/drive


### Build the clip list and convert .MOV → .mp4
iPhone .MOV files are often HEVC, which Colab's video readers can't always decode, so each clip is re-encoded to H.264 at 480p (also makes inference faster).

In [5]:
import re, subprocess
import pandas as pd

rows = []
for f in sorted(os.listdir(CLIP_DIR)):
    if not f.lower().endswith(".mov"):
        continue
    stem = f.rsplit(".", 1)[0]                       # e.g. bottle_wrongpocket2
    task_key, rest = stem.split("_", 1)              # bottle, wrongpocket2
    if task_key not in TASKS:
        print("skipping (unknown task):", f); continue
    ftype = re.sub(r"\d+$", "", rest)               # wrongpocket
    label = "success" if ftype == "success" else "failure"
    mp4 = stem + ".mp4"
    dst = os.path.join(MP4_DIR, mp4)
    if not os.path.exists(dst):
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", os.path.join(CLIP_DIR, f),
                        "-vf", "scale=-2:480", "-c:v", "libx264", "-preset", "fast", "-an", dst],
                       check=True)
    task, objects = TASKS[task_key]
    rows.append({"filename": mp4, "category": task_key, "failure_type": ftype,
                 "task": task, "label": label, "objects": objects})

df = pd.DataFrame(rows)
df.to_csv(CSV_PATH, index=False)
print(df.groupby(["category", "label"]).size())
df

category  label  
bottle    failure    6
          success    3
keys      failure    3
          success    1
dtype: int64


,filename,category,failure_type,task,label,objects
0,bottle_ontopfell1.mp4,bottle,ontopfell,put the water bottle in the backpack's outermo...,failure,water bottle. backpack. pocket.
1,bottle_rightpocketdropped1.mp4,bottle,rightpocketdropped,put the water bottle in the backpack's outermo...,failure,water bottle. backpack. pocket.
2,bottle_success1.mp4,bottle,success,put the water bottle in the backpack's outermo...,success,water bottle. backpack. pocket.
3,bottle_success2.mp4,bottle,success,put the water bottle in the backpack's outermo...,success,water bottle. backpack. pocket.
4,bottle_success3.mp4,bottle,success,put the water bottle in the backpack's outermo...,success,water bottle. backpack. pocket.
5,bottle_wrongpocket1.mp4,bottle,wrongpocket,put the water bottle in the backpack's outermo...,failure,water bottle. backpack. pocket.
6,bottle_wrongpocket2.mp4,bottle,wrongpocket,put the water bottle in the backpack's outermo...,failure,water bottle. backpack. pocket.
7,bottle_wrongpocket3.mp4,bottle,wrongpocket,put the water bottle in the backpack's outermo...,failure,water bottle. backpack. pocket.
8,bottle_wrongpocketdropped1.mp4,bottle,wrongpocketdropped,put the water bottle in the backpack's outermo...,failure,water bottle. backpack. pocket.
9,keys_lanyardvisible1.mp4,keys,lanyardvisible,put the keys inside the backpack and zip it co...,failure,keys. lanyard. backpack. zipper.


## 2. Load models

In [6]:
import torch, time, json, re, os, cv2
import pandas as pd
from PIL import Image
from transformers import (Qwen2_5_VLForConditionalGeneration, AutoProcessor,
                          AutoModelForZeroShotObjectDetection)
from qwen_vl_utils import process_vision_info

DEVICE = "cuda"
vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_ID, torch_dtype=torch.float16, device_map="auto", attn_implementation="sdpa")
vlm_proc = AutoProcessor.from_pretrained(VLM_ID)

dino = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_ID).to(DEVICE)
dino_proc = AutoProcessor.from_pretrained(DINO_ID)
print("models loaded")

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

models loaded


## 3. Helpers: Video-LLM calls, JSON parsing, call counter

In [7]:
CALLS = {"n": 0}

def _generate(messages, max_new_tokens=256):
    CALLS["n"] += 1
    text = vlm_proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    try:
        imgs, vids, vkw = process_vision_info(messages, return_video_kwargs=True)
    except TypeError:
        imgs, vids = process_vision_info(messages); vkw = {}
    inputs = vlm_proc(text=[text], images=imgs, videos=vids, padding=True,
                      return_tensors="pt", **vkw).to(vlm.device)
    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    out = out[:, inputs.input_ids.shape[1]:]
    return vlm_proc.batch_decode(out, skip_special_tokens=True)[0].strip()

def ask_video(path, prompt, max_new_tokens=256):
    msgs = [{"role": "user", "content": [
        {"type": "video", "video": f"file://{path}", "fps": FPS, "max_pixels": MAX_PIXELS},
        {"type": "text", "text": prompt}]}]
    return _generate(msgs, max_new_tokens)

def ask_text(prompt, max_new_tokens=256):
    msgs = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    return _generate(msgs, max_new_tokens)

def parse_json(s, default):
    m = re.search(r"(\{.*\}|\[.*\])", s, re.S)
    if not m:
        return default
    try:
        return json.loads(m.group(1))
    except Exception:
        return default

def to_bool_success(v):
    if isinstance(v, bool): return v
    return str(v).strip().lower() in {"true", "yes", "success", "1"}

## 4. Object-centric summary with Grounding DINO

In [8]:
def sample_frames(path, n):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS) or 30
    idxs = [int(i * (total - 1) / max(n - 1, 1)) for i in range(n)]
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i); ok, f = cap.read()
        if ok:
            frames.append((i / fps, Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))))
    cap.release()
    return frames

def object_summary(path, objects):
    lines = []
    for t, img in sample_frames(path, DINO_FRAMES):
        inp = dino_proc(images=img, text=objects, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = dino(**inp)
        kw = dict(target_sizes=[img.size[::-1]], text_threshold=0.25)
        try:
            res = dino_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=0.35, **kw)[0]
        except TypeError:
            res = dino_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=0.35, **kw)[0]
        W, H = img.size
        labels = res.get("text_labels", res.get("labels"))
        dets = []
        for lab, box, sc in zip(labels, res["boxes"].tolist(), res["scores"].tolist()):
            cx, cy = (box[0] + box[2]) / 2 / W, (box[1] + box[3]) / 2 / H
            dets.append(f"{lab}@center({cx:.2f},{cy:.2f}) conf {sc:.2f}")
        lines.append(f"t={t:.1f}s: " + ("; ".join(dets) if dets else "no target objects detected"))
    return "\n".join(lines)

## 5. Baseline: one-shot Video-LLM

In [9]:
def baseline(path, task):
    prompt = (f"A robot was asked to: '{task}'. Watch the video. Did it fully succeed? "
              'Reply ONLY with JSON: {"success": true/false, "failure_time_s": number or null, '
              '"correction": "one short instruction to fix it, or empty"}')
    return parse_json(ask_video(path, prompt), {"success": None})

## 6. DIVE-inspired method
Intent estimation → question breakdown → object summary → iterative QA + continuation judgment → verdict → verification.

In [10]:
def dive_lite(path, task, objects=None):
    trace = {}
    # 1. Intent estimation: turn the instruction into a checkable success condition
    cond = ask_text(f"Task given to a robot: '{task}'. In one sentence, state the exact visual "
                    "condition that must be true at the END of the video for the task to count as fully done.")
    trace["condition"] = cond

    # objects for Grounding DINO
    if not objects:
        objs = parse_json(ask_text(f"List the physical objects involved in: '{task}'. "
                                   'Reply ONLY with a JSON list of short nouns.'), [])
        objects = ". ".join(objs) + "." if objs else "object."
    summary = object_summary(path, objects)
    trace["object_summary"] = summary

    # 2. Question breakdown
    subqs = parse_json(ask_text(
        f"Success condition: {cond}\nWrite 2-4 short yes/no questions that can be checked by "
        "looking at the video and that together decide whether the condition holds. "
        "Include at least one question about the final frames. Reply ONLY with a JSON list of strings."), [])
    if not subqs:
        subqs = [f"At the end of the video, is this true: {cond}?"]

    # 3. Iterative QA with continuation judgment
    qa = []
    for rnd in range(MAX_ROUNDS):
        for q in subqs:
            a = ask_video(path, f"Object detections over time (may be noisy):\n{summary}\n\n"
                                f"Question: {q}\nLook carefully, especially at the last frames. "
                                "Answer yes or no, then one sentence of visual evidence.", 96)
            qa.append((q, a))
        evidence = "\n".join(f"Q: {q}\nA: {a}" for q, a in qa)
        judge = parse_json(ask_text(
            f"Success condition: {cond}\nEvidence so far:\n{evidence}\n"
            "Is this enough to decide success vs failure? "
            'Reply ONLY with JSON: {"sufficient": true/false, "new_questions": [..]}'),
            {"sufficient": True})
        if judge.get("sufficient", True) or not judge.get("new_questions"):
            break
        subqs = judge["new_questions"][:2]
    trace["qa"] = qa

    # 4. Verdict
    verdict = parse_json(ask_video(path,
        f"Task: '{task}'. Success condition: {cond}\nEvidence:\n{evidence}\n"
        'Decide. Reply ONLY with JSON: {"success": true/false, "failure_time_s": number or null, '
        '"correction": "one short instruction to fix it, or empty"}'), {"success": None})

    # 5. Verification (two-stage check against the video)
    check = parse_json(ask_video(path,
        f"Task: '{task}'. A previous answer said: {json.dumps(verdict)}. "
        "Re-watch the video, especially the final frames. Does that answer contradict what is visible? "
        'Reply ONLY with JSON: {"is_valid": true/false, "success": true/false, "correction": "..."}'),
        {"is_valid": True})
    if not check.get("is_valid", True) and "success" in check:
        verdict["success"] = check["success"]
        verdict["correction"] = check.get("correction", verdict.get("correction", ""))
        verdict["revised_by_verifier"] = True
    trace["verdict"] = verdict
    return verdict, trace

## 7. Run on all clips

In [11]:
df = pd.read_csv(CSV_PATH).fillna("")
rows, traces = [], {}
for _, r in df.iterrows():
    path = os.path.join(MP4_DIR, r["filename"])
    gt_success = r["label"].strip().lower() == "success"
    for method in ["baseline", "dive_lite"]:
        CALLS["n"] = 0; t0 = time.time()
        if method == "baseline":
            out = baseline(path, r["task"])
        else:
            out, tr = dive_lite(path, r["task"], r.get("objects") or None)
            traces[r["filename"]] = tr
        pred = to_bool_success(out.get("success")) if out.get("success") is not None else None
        rows.append({"filename": r["filename"], "category": r["category"],
                     "failure_type": r["failure_type"], "method": method,
                     "gt_success": gt_success, "pred_success": pred,
                     "correct": pred == gt_success,
                     "failure_time_s": out.get("failure_time_s"),
                     "correction": out.get("correction", ""),
                     "calls": CALLS["n"], "seconds": round(time.time() - t0, 1)})
        print(r["filename"], method, "pred:", pred, "gt:", gt_success, f"({rows[-1]['seconds']}s)")

res = pd.DataFrame(rows); res.to_csv(OUT_CSV, index=False)
json.dump(traces, open(OUT_CSV.replace(".csv", "_traces.json"), "w"), indent=2, default=str)

qwen-vl-utils using torchcodec to read video.


StrictDataclassFieldValidationError: Validation error for field 'fps':
    TypeError: Field 'fps' with value [0.8421052631578948] doesn't match any type in (<class 'int'>, <class 'float'>, <class 'NoneType'>). Errors: Field 'fps' expected int, got list (value: [0.8421052631578948]); Field 'fps' expected float, got list (value: [0.8421052631578948]); Field 'fps' expected NoneType, got list (value: [0.8421052631578948])

## 8. Results table

In [ ]:
def failure_f1(g):
    tp = ((g.pred_success == False) & (g.gt_success == False)).sum()
    fp = ((g.pred_success == False) & (g.gt_success == True)).sum()
    fn = ((g.pred_success != False) & (g.gt_success == False)).sum()
    return 2 * tp / max(2 * tp + fp + fn, 1)

by_cat = res.pivot_table(index="category", columns="method", values="correct", aggfunc="mean").round(2)
overall = res.groupby("method").apply(lambda g: pd.Series({
    "accuracy": g.correct.mean(), "failure_F1": failure_f1(g),
    "avg_calls": g.calls.mean(), "avg_seconds": g.seconds.mean(),
    "missed_failures(said success)": ((g.pred_success == True) & (g.gt_success == False)).sum()})).round(2)
print("Accuracy by failure category:\n", by_cat, "\n")
print("Overall:\n", overall)

### Per-clip results

In [ ]:
per_clip = res.pivot_table(index=["category", "failure_type", "filename"], columns="method",
                           values="correct", aggfunc="first").replace({True: "✓", False: "✗"})
per_clip["dive_lite_seconds"] = res[res.method == "dive_lite"].set_index("filename")["seconds"].reindex(
    per_clip.index.get_level_values("filename")).values
per_clip.to_csv(f"{OUT_DIR}/per_clip_results.csv")
per_clip

In [ ]:
import matplotlib.pyplot as plt
ax = by_cat.plot(kind="bar", figsize=(7, 4), ylim=(0, 1), rot=20)
ax.set_ylabel("Detection accuracy"); ax.set_title("One-shot vs DIVE-lite by failure category")
plt.tight_layout(); plt.savefig(OUT_CSV.replace(".csv", "_chart.png"), dpi=200); plt.show()

## 9. Inspect one trace

In [ ]:
name = list(traces)[0]   # change to any filename
t = traces[name]
print("CONDITION:", t["condition"], "\n\nOBJECTS:\n", t["object_summary"], "\n")
for q, a in t["qa"]: print("Q:", q, "\nA:", a, "\n")
print("VERDICT:", t["verdict"])